<!-- Problem Understanding Header -->
<h1 id="problem" style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 14px 26px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: bold;
  text-align: center;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
🧬 Predicting Protein Function 
</h1>

<p style="margin-bottom: 10px; font-weight: 600;">
      📌 What is the task?
    </p>
    <p style="margin-bottom: 10px;">
      This competition asks us to predict Gene Ontology (GO) terms for proteins based on their sequences. GO terms are standardized labels that describe:

<ul style="padding-left: 20px; margin-bottom: 0;">
      <li style="margin-bottom: 6px;">What molecular job the protein performs  </li>
      <li style="margin-bottom: 6px;">Which biological processes it's involved in</li>
      <li style="margin-bottom: 6px;">Where in the cell it operates  </li>
    </ul>
    <p style="margin-top: 10px;">
     A single protein can have multiple functions, so this is a **multi-label classification** problem. We're not picking one category — we're predicting all the relevant functional labels simultaneously.
    </p>
    </p>


 



</div>


In [1]:
!pip install obonet 

<!-- Section Header with Simple Numbering -->
<h1 id="setup-and-configuration"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  1. Importing Libraries
</h1>


In [2]:
import numpy as np
import pandas as pd
import obonet
import torch
from torch import nn
import networkx as nx
from torch.utils.data import Dataset, DataLoader, random_split
from torchmetrics.classification import MultilabelF1Score
from sklearn.preprocessing import MultiLabelBinarizer
from matplotlib import pyplot as plt
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

<!-- Section Header with Simple Numbering -->
<h1 id="configuration"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  2. Configuration
</h1>


In [3]:
class Config:
    """Central configuration for all hyperparameters and paths"""
    
    # Data paths
    train_embeds_path = "/kaggle/input/cafa-6-protbert-embeddings-dataset/train_embeds_final.npy"
    test_embeds_path  = "/kaggle/input/cafa-6-protbert-embeddings-dataset/test_embeds_final.npy"
    train_ids_path   = "/kaggle/input/cafa-6-protbert-embeddings-dataset/train_ids_final.npy"
    test_ids_path    = "/kaggle/input/cafa-6-protbert-embeddings-dataset/test_ids_final.npy"
    train_terms_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv"
    obo_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"

    # Model architecture
    input_dim = 1024      
    num_labels = 200     
    hidden_dims = [1024, 512, 512]  
    dropout = 0.3         

    # Training parameters
    batch_size = 128
    train_split = 0.9    
    lr = 1e-3             
    epochs = 100
    patience = 15         

    # Device configuration
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 42

cfg = Config()
print(f"Using device: {cfg.device}")


Using device: cuda


<!-- Section Header with Simple Numbering -->
<h1 id="data-load"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  3. Data Loading
</h1>


In [4]:
print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

# Load pre-computed ProtBERT embeddings
train_embeds = np.load(cfg.train_embeds_path)
test_embeds  = np.load(cfg.test_embeds_path)
train_ids    = np.load(cfg.train_ids_path)
test_ids     = np.load(cfg.test_ids_path)

print(f"Train embeddings shape: {train_embeds.shape}")
print(f"Test embeddings shape:  {test_embeds.shape}")

# Load GO term labels
train_terms_df = pd.read_csv(cfg.train_terms_path, sep="\t")

# Group GO terms by protein ID
labels_df = (
    train_terms_df
    .groupby("EntryID")["term"]
    .apply(list)
    .reset_index(name="labels")
)

# Select top N most frequent GO terms
top_go = (
    train_terms_df["term"]
    .value_counts()
    .iloc[:cfg.num_labels]
    .index
    .tolist()
)

print(f"\nUnique proteins with labels: {len(labels_df)}")
print(f"Top GO terms selected: {len(top_go)}")


LOADING DATA
Train embeddings shape: (82404, 1024)
Test embeddings shape:  (224309, 1024)

Unique proteins with labels: 82404
Top GO terms selected: 200


<!-- Section Header with Simple Numbering -->
<h1 id="configuration"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  4. Data Preprocessing
</h1>


In [5]:
go_graph = obonet.read_obo(cfg.obo_path)

def get_ancestors(graph, term):
    if term not in graph:
        return set()
    return nx.descendants(graph, term) | {term}

print("\n" + "="*60)
print("PROPAGATING GO TERM ANCESTORS")
print("="*60)

# Select top N most frequent GO terms
top_go = (
    train_terms_df["term"]
    .value_counts()
    .iloc[:cfg.num_labels]
    .index
    .tolist()
)
top_go_set = set(top_go)

# Group GO terms by protein ID
protein_to_terms = train_terms_df.groupby("EntryID")["term"].apply(set).to_dict()

# Propagate ancestors for each protein
propagated_labels = {}
print("Propagating ancestors for each protein...")

for protein_id, terms in tqdm(protein_to_terms.items(), desc="Propagating"):
    all_ancestors = set()
    for term in terms:
        # Get all ancestors for this term
        ancestors = get_ancestors(go_graph, term)
        all_ancestors.update(ancestors)
    
    # Filter to only keep terms in our top N list
    propagated_labels[protein_id] = all_ancestors.intersection(top_go_set)

# Create labels DataFrame with propagated terms
labels_data = []
for protein_id, terms in propagated_labels.items():
    if terms:  # Only include proteins with at least one term
        labels_data.append({
            "EntryID": protein_id,
            "labels": list(terms)
        })

labels_df = pd.DataFrame(labels_data)


print("\n" + "="*60)
print("PREPROCESSING DATA")
print("="*60)

# Initialize MultiLabelBinarizer for one-hot encoding GO terms
mlb = MultiLabelBinarizer(classes=top_go)
mlb.fit([top_go])

# Convert GO term lists to binary target vectors
labels_df["targets"] = list(mlb.transform(labels_df["labels"]))

# Create training DataFrame with embeddings and labels
train_emb_df = pd.DataFrame({
    "EntryID": train_ids,
    "embed": list(train_embeds)
})

train_df = train_emb_df.merge(labels_df, on="EntryID", how="inner")

# Create test DataFrame
test_df = pd.DataFrame({
    "EntryID": test_ids,
    "embed": list(test_embeds)
})

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")


PROPAGATING GO TERM ANCESTORS
Propagating ancestors for each protein...


Propagating: 100%|██████████| 82404/82404 [00:11<00:00, 7037.43it/s]



PREPROCESSING DATA
Training data shape: (74429, 4)
Test data shape: (224309, 2)


<!-- Section Header with Simple Numbering -->
<h1 id="model-def"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  5. Model Definition
</h1>


In [6]:
class MLP(nn.Module):
    
    def __init__(self, input_dim=1024, num_labels=500, hidden_dims=[1024, 512, 512], dropout=0.3):
        super().__init__()
        self.layers = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.dropouts = nn.ModuleList()

        # Build hidden layers
        prev_dim = input_dim
        for h in hidden_dims:
            self.layers.append(nn.Linear(prev_dim, h))
            self.bns.append(nn.BatchNorm1d(h))
            self.dropouts.append(nn.Dropout(dropout))
            prev_dim = h

        # Output layer
        self.out = nn.Linear(prev_dim, num_labels)

    def forward(self, x):
        residual = x
        
        # Pass through hidden layers with residual connections
        for i, layer in enumerate(self.layers):
            x = layer(x)
            x = self.bns[i](x)
            x = torch.relu(x)
            x = self.dropouts[i](x)
            
            # Add residual connection every 2 layers
            if i % 2 == 1:
                if residual.shape[1] == x.shape[1]:
                    x = x + residual
                residual = x
            else:
                residual = x
        
        # Output layer
        x = self.out(x)
        return x

<!-- Section Header with Simple Numbering -->
<h1 id="dataset"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  6. Dataset and DataLoaders
</h1>


In [7]:
class ProteinFunctionDataset(Dataset):
    """Custom Dataset for protein embeddings and GO term labels"""
    
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        embed = torch.tensor(self.df.loc[idx, "embed"], dtype=torch.float32)
        
        if self.train:
            target = torch.tensor(self.df.loc[idx, "targets"], dtype=torch.float32)
            return embed, target
        else:
            entry_id = self.df.loc[idx, "EntryID"]
            return embed, entry_id

# Create datasets
full_train_dataset = ProteinFunctionDataset(train_df, train=True)
test_dataset = ProteinFunctionDataset(test_df, train=False)

# Split training data into train and validation sets
train_size = int(len(full_train_dataset) * cfg.train_split)
val_size = len(full_train_dataset) - train_size

g = torch.Generator().manual_seed(cfg.seed)
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=g)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")



Train batches: 524
Validation batches: 59
Test batches: 1753


<!-- Section Header with Simple Numbering -->
<h1 id="training"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  7. Training loop
</h1>


In [8]:
def train_model(model, train_loader, val_loader, cfg, save_path="best_model.pt"):
  
    
    model.to(cfg.device)

    # Initialize optimizer, scheduler, and loss function
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )
    loss_fn = nn.CrossEntropyLoss()
    # loss_fn = nn.BCEWithLogitsLoss() 
    f1_score = MultilabelF1Score(num_labels=cfg.num_labels).to(cfg.device)

    # Tracking metrics
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_f1": [],
        "val_f1": []
    }

    best_val_loss = float("inf")
    patience_counter = 0

    print("\n" + "="*60)
    print("TRAINING")
    print("="*60)

    for epoch in range(cfg.epochs):
        # ===== TRAINING PHASE =====
        model.train()
        train_losses, train_f1s = [], []

        for embeds, targets in train_loader:
            embeds = embeds.to(cfg.device)
            targets = targets.to(cfg.device)

            # Forward pass
            optimizer.zero_grad()
            logits = model(embeds)
            loss = loss_fn(logits, targets)
            
            # Backward pass
            loss.backward()
            optimizer.step()

            # Calculate metrics
            f1 = f1_score(logits, targets)
            train_losses.append(loss.item())
            train_f1s.append(f1.item())

        avg_train_loss = np.mean(train_losses)
        avg_train_f1 = np.mean(train_f1s)

        # ===== VALIDATION PHASE =====
        model.eval()
        val_losses, val_f1s = [], []

        with torch.no_grad():
            for embeds, targets in val_loader:
                embeds = embeds.to(cfg.device)
                targets = targets.to(cfg.device)

                logits = model(embeds)
                loss = loss_fn(logits, targets)

                f1 = f1_score(logits, targets)
                val_losses.append(loss.item())
                val_f1s.append(f1.item())

        avg_val_loss = np.mean(val_losses)
        avg_val_f1 = np.mean(val_f1s)

        # Save metrics
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["train_f1"].append(avg_train_f1)
        history["val_f1"].append(avg_val_f1)

        # Print progress
        print(
            f"Epoch {epoch+1:3d}/{cfg.epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
            f"Train F1: {avg_train_f1:.4f} | Val F1: {avg_val_f1:.4f}"
        )

        # Learning rate scheduling
        scheduler.step(avg_val_loss)

        # ===== CHECKPOINTING & EARLY STOPPING =====
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f"💾 Best model saved at epoch {epoch+1}")
        else:
            patience_counter += 1
            print(f"⏳ No improvement. Patience: {patience_counter}/{cfg.patience}")

        if patience_counter >= cfg.patience:
            print("🛑 Early stopping triggered.")
            break

    print(f"\n✅ Training complete! Best Val Loss: {best_val_loss:.4f}")
    return history


model = MLP(
    input_dim=cfg.input_dim,
    num_labels=cfg.num_labels,
    hidden_dims=cfg.hidden_dims,
    dropout=cfg.dropout
)

history = train_model(model, train_loader, val_loader, cfg, save_path="best_model.pt")



TRAINING
Epoch   1/100 | Train Loss: 25.1978 | Val Loss: 24.5138 | Train F1: 0.1213 | Val F1: 0.1481
💾 Best model saved at epoch 1
Epoch   2/100 | Train Loss: 24.1528 | Val Loss: 24.1404 | Train F1: 0.1495 | Val F1: 0.1566
💾 Best model saved at epoch 2
Epoch   3/100 | Train Loss: 23.8375 | Val Loss: 24.0326 | Train F1: 0.1576 | Val F1: 0.1581
💾 Best model saved at epoch 3
Epoch   4/100 | Train Loss: 23.6104 | Val Loss: 23.8476 | Train F1: 0.1638 | Val F1: 0.1679
💾 Best model saved at epoch 4
Epoch   5/100 | Train Loss: 23.4289 | Val Loss: 23.6873 | Train F1: 0.1684 | Val F1: 0.1707
💾 Best model saved at epoch 5
Epoch   6/100 | Train Loss: 23.2979 | Val Loss: 23.6129 | Train F1: 0.1724 | Val F1: 0.1742
💾 Best model saved at epoch 6
Epoch   7/100 | Train Loss: 23.1670 | Val Loss: 23.5465 | Train F1: 0.1758 | Val F1: 0.1755
💾 Best model saved at epoch 7
Epoch   8/100 | Train Loss: 23.0428 | Val Loss: 23.5126 | Train F1: 0.1791 | Val F1: 0.1793
💾 Best model saved at epoch 8
Epoch   9/100 

<!-- Section Header with Simple Numbering -->
<h1 id="submission"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  8. Probert Submission 
</h1>


In [9]:
# Load the best model checkpoint saved during training
model.load_state_dict(torch.load("best_model.pt", map_location=cfg.device))
print("✅ Best model loaded successfully!")

def generate_submission(
    model,
    test_loader,
    go_terms,
    device,
    output_path="submission.tsv",
    max_terms_per_protein=1500,
    min_prob=1e-6
):
   
    
    model.eval()
    
    all_ids = []
    all_probs = []

    # Generate predictions
    with torch.no_grad():
        for embeds, ids in tqdm(test_loader, desc="Predicting"):
            embeds = embeds.to(device)
            logits = model(embeds)
            probs = torch.sigmoid(logits).cpu().numpy()

            all_probs.append(probs)
            all_ids.extend(ids)

    all_probs = np.vstack(all_probs)  # Shape: (N_test, num_labels)
    all_ids = np.array(all_ids)

    # Format predictions for submission
    rows = []

    for i, prot_id in enumerate(all_ids):
        probs = all_probs[i]

        # Keep only positive probabilities
        idx = np.where(probs > 0)[0]
        if len(idx) == 0:
            continue

        # Sort by confidence (descending)
        idx = idx[np.argsort(probs[idx])[::-1]]

        # Limit to top 1500 terms per protein 
        idx = idx[:max_terms_per_protein]

        for j in idx:
            p = probs[j]
            if p <= 0:
                continue

            # Format to 3 significant figures 
            p_fmt = float(f"{p:.3g}")
            if p_fmt <= 0:
                p_fmt = min_prob

            rows.append([prot_id, go_terms[j], p_fmt])

    # Create submission DataFrame
    sub_df = pd.DataFrame(rows)
    sub_df.to_csv(output_path, sep="\t", header=False, index=False)

    print(f"\n✅ Submission saved to {output_path}")
    print(f"Total predictions: {len(sub_df):,}")

    return sub_df

print("\n" + "="*60)
print("GENERATING SUBMISSION")
print("="*60)

submission_df = generate_submission(
    model=model,
    test_loader=test_loader,
    go_terms=top_go,
    device=cfg.device,
    output_path="submission.tsv"
)

# Display sample predictions
print("\nSample predictions:")
print(submission_df.head(10))

print("\n" + "="*60)
print("DONE!")
print("="*60)

✅ Best model loaded successfully!

GENERATING SUBMISSION


Predicting: 100%|██████████| 1753/1753 [00:06<00:00, 268.33it/s]



✅ Submission saved to submission.tsv
Total predictions: 44,861,800

Sample predictions:
            0           1      2
0  A0A0C5B5G6  GO:0010467  0.923
1  A0A0C5B5G6  GO:0010468  0.921
2  A0A0C5B5G6  GO:0006355  0.909
3  A0A0C5B5G6  GO:0005515  0.882
4  A0A0C5B5G6  GO:0005737  0.869
5  A0A0C5B5G6  GO:0005634  0.866
6  A0A0C5B5G6  GO:0005576  0.846
7  A0A0C5B5G6  GO:0045893  0.828
8  A0A0C5B5G6  GO:0007165  0.744
9  A0A0C5B5G6  GO:0006357  0.724

DONE!


<!-- Section Header with Simple Numbering -->
<h1 id="ensembling"
    style="
  background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
  color: white;
  padding: 12px 24px;
  text-align: center;
  border-radius: 12px;
  font-size: 22px;
  font-weight: bold;
  font-family: Arial, sans-serif;
  margin: 20px 0;
">
  9. Ensembling with blast 
</h1>


In [10]:
# import pandas as pd
# import numpy as np


# submission2 = pd.read_csv(
#     '/kaggle/input/blast-quick-sprof-zero-pred/submission.tsv',
#     sep='\t',
#     header=None,
#     names=['Id', 'GO term', 'Confidence2'],
#     dtype={'Id': 'category', 'GO term': 'category', 'Confidence2': 'float32'}
# )

# submission_df.columns = ['Id', 'GO term', 'Confidence']
# submission_df['Confidence'] = submission_df['Confidence'].astype('float32')

# subs = submission_df.merge(
#     submission2,
#     on=['Id', 'GO term'],
#     how='left'
# )

# subs['confidence_combined'] = subs['Confidence2'].combine_first(subs['Confidence'])

# subs[['Id', 'GO term', 'confidence_combined']].to_csv(
#     'submission.tsv',
#     sep='\t',
#     header=False,
#     index=False
# )
